# Genre Based Model 


In [1]:
#ingest downloaded csvs
import pandas as pd
movies = pd.read_csv('movies.csv')
principals  = pd.read_csv('principals.csv')
ratings = pd.read_csv('ratings.csv')
plotline = pd.read_csv('plotline.csv')
names = pd.read_csv('names.csv')

### 1. Vectorization

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

#-------------------- sanjana's change ---------------------
#cleaned up Genres Column for the CountVectorizer
movies['Genre'] = movies['Genre'].fillna('')
movies['Genre'] = movies['Genre'].str.replace(' ', '')
movies['Genre'] = movies['Genre'].str.lower()
#-------------------- sanjana's change ---------------------


#Change Genre list to pipe-separated string for CountVectorizer
movies['Genre'] = movies['Genre'].apply(lambda x: '|'.join(x) if isinstance(x, list) else x)  

#-------------------- sanjana's change ---------------------

count_vectorizer = CountVectorizer(tokenizer = lambda x: x.split('|'), token_pattern = None, binary = True)

genre_matrix = count_vectorizer.fit_transform(movies['Genre'])
#-------------------- sanjana's change ---------------------


genre_features = pd.DataFrame(
    genre_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=movies['Title']
)

print(f'Genres list: {list(count_vectorizer.get_feature_names_out())}')
genre_features.head()

Genres list: ["['action','adventure','animation']", "['action','adventure','biography']", "['action','adventure','comedy']", "['action','adventure','crime']", "['action','adventure','drama']", "['action','adventure','family']", "['action','adventure','fantasy']", "['action','adventure','history']", "['action','adventure','horror']", "['action','adventure','mystery']", "['action','adventure','romance']", "['action','adventure','sci-fi']", "['action','adventure','thriller']", "['action','adventure','war']", "['action','adventure']", "['action','animation','comedy']", "['action','animation','crime']", "['action','animation','drama']", "['action','animation','family']", "['action','animation','fantasy']", "['action','animation','horror']", "['action','animation','music']", "['action','animation','sci-fi']", "['action','biography','comedy']", "['action','biography','crime']", "['action','biography','drama']", "['action','biography','history']", "['action','comedy','crime']", "['action','com

,"['action','adventure','animation']","['action','adventure','biography']","['action','adventure','comedy']","['action','adventure','crime']","['action','adventure','drama']","['action','adventure','family']","['action','adventure','fantasy']","['action','adventure','history']","['action','adventure','horror']","['action','adventure','mystery']",...,"['romance','thriller','war']","['romance','thriller']","['romance','war']","['romance','western']",['romance'],"['sci-fi','thriller']",['sci-fi'],"['thriller','war']",['thriller'],['western']
Title,,,,,,,,,,,,,,,,,,,,,
Dante's Inferno (L'inferno),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
The Student of Prague (Der Student von Prag),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
The Last Days of Pompeii (Gli ultimi giorni di Pompei),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Julius Caesar (Cajus Julius Caesar),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"His Majesty, the Scarecrow of Oz",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 2. Compute cosine similarity

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(genre_matrix)

similarity_df = pd.DataFrame(similarity, index=movies['Title'], columns=movies['Title'])

similarity_df.iloc[:5, :5]

Title,Dante's Inferno (L'inferno),The Student of Prague (Der Student von Prag),The Last Days of Pompeii (Gli ultimi giorni di Pompei),Julius Caesar (Cajus Julius Caesar),"His Majesty, the Scarecrow of Oz"
Title,,,,,
Dante's Inferno (L'inferno),1.0,0.0,0.0,0.0,0.0
The Student of Prague (Der Student von Prag),0.0,1.0,0.0,0.0,0.0
The Last Days of Pompeii (Gli ultimi giorni di Pompei),0.0,0.0,1.0,0.0,0.0
Julius Caesar (Cajus Julius Caesar),0.0,0.0,0.0,1.0,0.0
"His Majesty, the Scarecrow of Oz",0.0,0.0,0.0,0.0,1.0


### 3. genre recommendation function

In [4]:

def genre_recommend(title, n=10):
    
    # Check if title exists in similarity_df
    if title not in similarity_df.columns:
        print(f'Title not found: "{title}"')
        matches = [t for t in similarity_df.columns if title.lower() in t.lower()]
        print('Possible matches:', matches[:10])
        return None

    # for duplicate titles
    count = list(similarity_df.columns).count(title)
    if count > 1:
        # lookup row - the best one by rating
        best_idx = movies[movies['Title'] == title].sort_values('Weighted Rating', ascending=False).index[0]
        # movies row = similarity_df column position
        col_pos = movies.index.get_loc(best_idx)
        best = movies.loc[best_idx]
        print(f'Auto-selected: {best["Title"]} ({best["Release Year"]}) — Rating: {best["Weighted Rating"]}')
    else:
        col_pos = list(similarity_df.columns).index(title)

    #  similarity score 
    sim_df = pd.DataFrame({
        'Title': list(similarity_df.index),
        'Similarity': similarity_df.iloc[:, col_pos].values
    })

    # Exclude the input title and get top N similar movies
    top_n = sim_df[sim_df['Title'] != title].sort_values('Similarity', ascending=False).head(n)

    # Build results with movie metadata
    result = top_n.merge(
        movies[['Movie ID', 'Title', 'Genre', 'Release Year', 'Weighted Rating']],
        on='Title'
    )
    result = result.rename(columns={'Similarity': 'Genre Similarity Score'})
    result['Genre Similarity Score'] = result['Genre Similarity Score'].round(4)

    return result.sort_values('Weighted Rating', ascending=False).reset_index(drop=True)

## Test

In [ ]:
# Test 1: The Dark Knight
input1 = 'The Dark Knight'

print(f'Genre Recommendations for: {input1}')
recommend1 = genre_recommend(input1, n=10)
recommend1[['Movie ID', 'Title', 'Genre', 'Weighted Rating', 'Genre Similarity Score']]

Genre Recommendations for: Snow White
Auto-selected: Snow White (1987) — Rating: 6.5


,Movie ID,Title,Genre,Weighted Rating,Genre Similarity Score
0,tt0101414,Beauty and the Beast,"['animation','family','fantasy']",8.0,1.0
1,tt0099263,Jim Henson's: The Christmas Toy (The Christmas...,"['family','fantasy','musical']",7.7,1.0
2,tt0042332,Cinderella,"['animation','family','fantasy']",7.3,1.0
3,tt0053285,Sleeping Beauty,"['adventure','animation','family']",7.2,1.0
4,tt2771200,Beauty and the Beast,"['adventure','family','fantasy']",7.1,1.0
5,tt0302625,Hansel and Gretel,['musical'],7.1,1.0
6,tt0764662,Rumpelstiltskin,"['animation','family']",7.1,1.0
7,tt1202150,Beauty and the Beast,"['animation','family']",7.1,1.0
8,tt1380833,Hansel and Gretel,['music'],6.9,1.0
9,tt1661199,Cinderella,"['adventure','drama','family']",6.9,1.0


In [ ]:
# Test 2: A Nightmare on Elm Street
input2 = 'A Nightmare on Elm Street'

print(f' Genre Recommendations for: {input2}')
recommend2 = genre_recommend(input2, n=10)
recommend2[['Movie ID', 'Title', 'Genre', 'Weighted Rating', 'Genre Similarity Score']]

 Genre Recommendations for: A Nightmare on Elm Street
Auto-selected: A Nightmare on Elm Street (1984) — Rating: 7.4


,Movie ID,Title,Genre,Weighted Rating,Genre Similarity Score
0,tt0082235,Dark Night of the Scarecrow,['horror'],6.7,1.0
1,tt0070770,Tales That Witness Madness,['horror'],5.7,1.0
2,tt0076853,The Uncanny,['horror'],5.7,1.0
3,tt0083629,The Beast Within,['horror'],5.6,1.0
4,tt0070673,The Severed Arm,['horror'],5.1,1.0
5,tt0079855,Savage Weekend,['horror'],4.6,1.0
6,tt0091990,Sorority House Massacre,['horror'],4.5,1.0
7,tt0096808,Amityville Horror: The Evil Escapes,['horror'],4.5,1.0
8,tt0082237,Dawn of the Mummy,['horror'],4.1,1.0
9,tt0410332,August Underground's Mordum,['horror'],3.1,1.0


In [ ]:
# Test 3: 12 Angry Men
input3 = '12 Angry Men'

print(f' Genre Recommendations for: {input3}')
recommend3 = genre_recommend(input3, n=10)
recommend3[['Movie ID', 'Title', 'Genre', 'Weighted Rating', 'Genre Similarity Score']]

 Genre Recommendations for: 12 Angry Men
Auto-selected: 12 Angry Men (1957) — Rating: 9.0


,Movie ID,Title,Genre,Weighted Rating,Genre Similarity Score
0,tt0075314,Taxi Driver,"['crime','drama']",8.2,1.0
1,tt0112641,Casino,"['crime','drama']",8.2,1.0
2,tt0110729,Once Were Warriors,"['crime','drama']",7.9,1.0
3,tt0055618,Whistle Down the Wind,"['crime','drama']",7.6,1.0
4,tt1916728,Shor in the City,"['crime','drama']",7.2,1.0
5,tt0039947,The Upturned Glass,"['crime','drama']",6.9,1.0
6,tt0068341,The Valachi Papers,"['crime','drama']",6.4,1.0
7,tt0102989,Straight Out of Brooklyn,"['crime','drama']",5.9,1.0
8,tt0305973,Sonny,"['crime','drama']",5.6,1.0
9,tt0066286,The Rebel Rousers,"['crime','drama']",4.4,1.0
